# DoE analysis for SpATA-catalyzed carbohydrate amination

## Purpose

This notebook generates and analyzes a face-centered central composite design for optimizing SpATA-catalyzed amination of C6-oxidized galactose. It fits a quadratic response-surface model, performs model diagnostics and effects analysis, generates response-surface plots, and calculates yield, titer, and multi-objective economic optima.

## Important preservation note

This is an **annotated copy**. Markdown guidance has been added, but the five original code cells, their order, outputs, and notebook metadata have been preserved. No original code was edited.

## Before running

1. Use Google Colab or a Python environment that permits package installation.
2. Upload the experimental Excel workbook to the notebook working directory.
3. Confirm the factor names, ranges, units, and response definition described below.
4. Confirm that the Excel filename and column positions match your workbook.
5. Run the code cells from top to bottom.

> **Data-integrity warning:** If the experimental workbook cannot be read or contains too few results, the core code enters **demo mode** and generates placeholder response values. Do not use demo-mode outputs as experimental results.


## Core workflow: setup, design, data import, model fitting, plots, and yield optimization

The next code cell contains Sections §1–§10. Use the section headings in the code cell to locate the settings below.

### Where to define or change the experiment

Find **`§2 — DEFINE YOUR EXPERIMENT`** and edit only when the experimental design changes:

- **`FACTORS`**: factor names, lower and upper bounds, units, and whether a factor is log-scaled.
- **`RESPONSE_NAME`**: name of the measured response; currently `Yield (%)`.
- **`N_CENTER`**: number of replicated center points; currently 5.

Current factors:

| Factor | Low | High | Unit | Scale |
|---|---:|---:|---|---|
| Donor | 1 | 7 | equivalents | linear |
| Acceptor | 5 | 50 | mM | linear |
| PLP | 0.001 | 1.0 | mM | logarithmic |
| Enzyme | 0.1 | 1.0 | mg mL⁻¹ | linear |

Do not edit the derived coding/decoding functions unless the design logic itself must change.

### Where the design matrix is created

**§3** creates the randomized face-centered CCD and exports `CCD_design_matrix.csv`. For a new experiment, fill the response column after completing the experimental runs.

### Where to provide experimental results

Find **`§4 — ENTER YOUR RESULTS`**:

- Change **`EXCEL_FILENAME`** from `DoE2_2.xlsx` to the exact name of the uploaded results workbook.
- The line `uploaded_df.iloc[:, [1, 3, 4, 5, 11]]` selects columns by zero-based position.
- Those columns are renamed to `Donor`, `Acceptor`, `PLP`, `Enzyme`, and the response name.
- If your spreadsheet layout changes, update both the column positions and, if necessary, the renamed column labels.
- Yield values in the range 0–1 are automatically converted to percentages.

The merge uses rounded physical factor values rather than run order. After loading, inspect the displayed mapping and `Merge_Verification.csv` before interpreting the model.

### Model and diagnostic sections

- **§5:** fits the full quadratic response-surface model and performs ANOVA.
- **§6:** generates residual and model-diagnostic plots.
- **§7:** evaluates main, quadratic, and interaction effects.
- **§8:** generates static response surfaces, contours, and perturbation plots.

### Where to change the yield-optimization objective

Find **`§9 — OPTIMISATION`**:

- **`OPTIMISE_FOR`** may be `maximise`, `minimise`, or `target`.
- **`TARGET_VALUE`** is used only when `OPTIMISE_FOR = 'target'`.
- The optimization bounds are the coded design space from −1 to +1. Changing the factor ranges in §2 changes the physical optimization bounds.
- The differential-evolution random seed is fixed at 42 for reproducibility.

### Main outputs

The cell exports the design matrix, merge-verification table, summary report, diagnostic plots, effects analysis, response surfaces, contours, perturbation plot, and optimum-location plot to the working directory.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#   RESPONSE SURFACE METHODOLOGY — FACE-CENTRED CCD
#   Google Colab Script
#   ─────────────────────────────────────────────────────────────────────────
#   HOW TO USE
#   ① Run §1  – installs packages, imports
#   ② Edit §2 – define YOUR factors and ranges (the only block you must edit)
#   ③ Run §3  – generates and exports the CCD matrix as CSV
#   ④ Run your experiments, fill results into the CSV
#   ⑤ Edit §4 – paste / upload your results
#   ⑥ Run §5–§8 – full RSM analysis, plots, and optimisation
# ══════════════════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════════════════
# §1 — SETUP
# ══════════════════════════════════════════════════════════════════════════════

# ── Install ───────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pyDOE2", "statsmodels", "seaborn", "-q"])

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize, differential_evolution
import statsmodels.api as sm
from statsmodels.formula.api import ols
from itertools import combinations
import warnings, io, os
warnings.filterwarnings("ignore")

!pip install pyDOE3
from pyDOE3 import ff2n
from IPython.display import display, HTML, FileLink
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

# ── Plot theme ────────────────────────────────────────────────────────────────
NAVY    = "#1F3864"
TEAL    = "#1D9E75"
AMBER   = "#EF9F27"
RED_C   = "#E24B4A"
SLATE   = "#5C6B7A"
BG      = "#F8F9FA"
CMAP_RS = LinearSegmentedColormap.from_list(
    "rsm_ryg", ["#C0392B","#E67E22","#F1C40F","#2ECC71","#1A8A4A"])

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "axes.edgecolor": SLATE, "axes.labelcolor": NAVY,
    "axes.titleweight": "bold", "axes.titlecolor": NAVY,
    "xtick.color": SLATE, "ytick.color": SLATE,
    "font.family": "DejaVu Sans", "font.size": 10,
    "legend.framealpha": 0.8, "legend.edgecolor": "#CCCCCC",
})

print("✓  Setup complete.")


# ══════════════════════════════════════════════════════════════════════════════
# §2 — DEFINE YOUR EXPERIMENT  ← UPDATED TO MATCH EXCEL
# ══════════════════════════════════════════════════════════════════════════════

FACTORS = {
    'Donor':    {'low': 1,     'high': 7,    'unit': 'eq',  'log_scale': False},
    'Acceptor': {'low': 5,     'high': 50,   'unit': 'mM',  'log_scale': False},
    'PLP':      {'low': 0.001, 'high': 1.0,  'unit': 'mM',  'log_scale': True },
    'Enzyme':   {'low': 0.1,   'high': 1.0,  'unit': 'mg/mL',  'log_scale': False},
}

RESPONSE_NAME  = 'Yield (%)'
N_CENTER       = 5

# ── Derived geometry (do not edit) ───────────────────────────────────────────
FACTOR_NAMES = list(FACTORS.keys())
K = len(FACTOR_NAMES)

def decode(coded_val, fname):
    """Convert coded value (−1, 0, +1) → real units."""
    f = FACTORS[fname]
    lo, hi = f['low'], f['high']
    if f['log_scale']:
        log_lo, log_hi = np.log10(lo), np.log10(hi)
        log_center = (log_lo + log_hi) / 2
        return 10 ** (log_center + coded_val * (log_hi - log_center))
    else:
        center = (lo + hi) / 2
        half   = (hi - lo) / 2
        return center + coded_val * half

def encode(real_val, fname):
    """Convert real value → coded (−1…+1)."""
    f = FACTORS[fname]
    lo, hi = f['low'], f['high']
    if f['log_scale']:
        log_lo, log_hi = np.log10(lo), np.log10(hi)
        log_ctr = (log_lo + log_hi) / 2
        half    = log_hi - log_ctr
        return (np.log10(real_val) - log_ctr) / half
    else:
        center = (lo + hi) / 2
        half   = (hi - lo) / 2
        return (real_val - center) / half

CENTERS = {fn: decode(0, fn) for fn in FACTOR_NAMES}

print(f"✓  {K} factors defined. Centre points (coded 0):")
for fn, cv in CENTERS.items():
    f = FACTORS[fn]
    ls = " [log-spaced]" if f['log_scale'] else ""
    print(f"   {fn:<12} {cv:.5g} {f['unit']}{ls}")

# ══════════════════════════════════════════════════════════════════════════════
# §3 — GENERATE FACE-CENTRED CCD MATRIX
# ══════════════════════════════════════════════════════════════════════════════

def build_fcc_matrix(factors, n_center=5):
    """
    Generate a face-centred CCD (α = 1) manually so there is no
    dependency on pyDOE2 for the geometry itself.
    Returns a DataFrame with coded and decoded columns.
    """
    fname_list = list(factors.keys())
    k = len(fname_list)

    # ── Factorial points ──────────────────────────────────────────────────────
    factorial_coded = np.array(
        [list(map(int, f"{i:0{k}b}".replace("0","-1 ").replace("1","1 ").split()))
         for i in range(2**k)]
    )
    # Simpler way: use itertools
    from itertools import product as iprod
    factorial_coded = np.array(list(iprod([-1, 1], repeat=k)), dtype=float)

    # ── Axial points (face-centred: α = 1) ───────────────────────────────────
    axial_coded = []
    for i in range(k):
        for sign in [-1, 1]:
            row = [0.0] * k
            row[i] = sign
            axial_coded.append(row)
    axial_coded = np.array(axial_coded)

    # ── Centre points ─────────────────────────────────────────────────────────
    center_coded = np.zeros((n_center, k))

    # ── Controls: no-enzyme and no-acceptor (if those factors exist) ──────────
    ctrl_rows = []
    no_enz = [0.0] * k
    no_acc = [0.0] * k
    if 'Enzyme' in fname_list:
        idx = fname_list.index('Enzyme')
        no_enz[idx] = -2.0          # sentinel for "zero" (will be replaced)
        ctrl_rows.append(('No_Enzyme_Control', no_enz))
    if 'Acceptor' in fname_list:
        idx = fname_list.index('Acceptor')
        no_acc[idx] = -2.0
        ctrl_rows.append(('No_Acceptor_Control', no_acc))

    # ── Assemble ──────────────────────────────────────────────────────────────
    rows = []
    run = 1

    for code in factorial_coded:
        row = {'Run': run, 'Run_Type': 'Factorial'}
        for i, fn in enumerate(fname_list):
            row[f'{fn}_coded'] = code[i]
            row[fn] = decode(code[i], fn)
        rows.append(row); run += 1

    for code in axial_coded:
        row = {'Run': run, 'Run_Type': 'Axial'}
        for i, fn in enumerate(fname_list):
            row[f'{fn}_coded'] = code[i]
            row[fn] = decode(code[i], fn)
        rows.append(row); run += 1

    for code in center_coded:
        row = {'Run': run, 'Run_Type': 'Centre'}
        for i, fn in enumerate(fname_list):
            row[f'{fn}_coded'] = 0.0
            row[fn] = decode(0, fn)
        rows.append(row); run += 1

    for ctype, code in ctrl_rows:
        row = {'Run': run, 'Run_Type': ctype}
        for i, fn in enumerate(fname_list):
            c = code[i]
            if c == -2.0:   # zero level (enzyme=0 or acceptor=0)
                row[f'{fn}_coded'] = -2.0
                row[fn] = 0.0
            else:
                row[f'{fn}_coded'] = c
                row[fn] = decode(c, fn)
        rows.append(row); run += 1

    df = pd.DataFrame(rows)
    # Randomise run order (keep controls at end)
    ctrl_mask  = df['Run_Type'].str.contains('Control')
    df_design  = df[~ctrl_mask].sample(frac=1, random_state=42).reset_index(drop=True)
    df_ctrl    = df[ctrl_mask].reset_index(drop=True)
    df_design['Run'] = range(1, len(df_design)+1)
    df_ctrl['Run']   = range(len(df_design)+1, len(df_design)+len(df_ctrl)+1)
    df_out = pd.concat([df_design, df_ctrl], ignore_index=True)

    # Add result column placeholder
    df_out[RESPONSE_NAME] = np.nan
    return df_out

DESIGN_DF = build_fcc_matrix(FACTORS, N_CENTER)

# ── Pretty display ────────────────────────────────────────────────────────────
coded_cols = [f'{fn}_coded' for fn in FACTOR_NAMES]
real_cols  = FACTOR_NAMES

print(f"\n{'='*70}")
print(f"  FACE-CENTRED CCD  |  {K} factors  |  {len(DESIGN_DF)} total runs")
print(f"  {2**K} factorial + {2*K} axial + {N_CENTER} centre + "
      f"{len(DESIGN_DF)-2**K-2*K-N_CENTER} controls")
print(f"{'='*70}\n")

display_df = DESIGN_DF[['Run','Run_Type'] + real_cols + coded_cols].copy()
for fn in FACTOR_NAMES:
    display_df[fn] = display_df[fn].map(lambda x: f"{x:.4g}")

# Colour-code by run type
def style_matrix(df):
    colors = {'Factorial':'#EAF3DE','Axial':'#E6F1FB','Centre':'#F1EFE8',
              'No_Enzyme_Control':'#FAEEDA','No_Acceptor_Control':'#FAEEDA'}
    def row_color(row):
        bg = colors.get(row['Run_Type'], 'white')
        return [f'background-color: {bg}'] * len(row)
    return df.style.apply(row_color, axis=1)

display(style_matrix(display_df))

# ── Export ────────────────────────────────────────────────────────────────────
DESIGN_DF.to_csv('CCD_design_matrix.csv', index=False)
print("\n✓  Design exported to  CCD_design_matrix.csv")
print("   → Fill in the '{}' column, then continue to §4.".format(RESPONSE_NAME))
try:
    display(FileLink('CCD_design_matrix.csv', result_html_prefix="⬇  Download: "))
except Exception:
    pass


# ══════════════════════════════════════════════════════════════════════════════
# §4 — ENTER YOUR RESULTS (SMART MERGE BY PHYSICAL CONDITIONS)
# ══════════════════════════════════════════════════════════════════════════════

EXCEL_FILENAME = 'DoE2_2.xlsx'

try:
    # 1. Read the Excel file
    uploaded_df = pd.read_excel(EXCEL_FILENAME)

    # 2. Extract ONLY the physical parameters and Yield based on your Excel layout
    # (Col 1: Donor eq, Col 3: Acceptor, Col 4: PLP, Col 5: Enzyme, Col 11: Yield)
    #subset_df = uploaded_df.iloc[:, [1, 3, 4, 5, 11]].copy()
    #subset_df.columns = ['Donor', 'Acceptor', 'PLP', 'Enzyme', RESPONSE_NAME]
    # 2. Extract ONLY the physical parameters and Yield based on your Excel layout
    # (Col 1: Donor eq, Col 3: Acceptor, Col 4: PLP, Col 5: Enzyme, Col 11: Yield)
    subset_df = uploaded_df.iloc[:, [1, 3, 4, 5, 11]].copy()
    subset_df.columns = ['Donor', 'Acceptor', 'PLP', 'Enzyme', RESPONSE_NAME]

    # ── ADD THIS DECIMAL FIX ──────────────────────────────────────────────────
    if subset_df[RESPONSE_NAME].max() <= 1.0:
        subset_df[RESPONSE_NAME] = subset_df[RESPONSE_NAME] * 100
        print("ℹ️ Auto-converted decimal yields (0-1) to percentages (0-100).")
    # ──────────────────────────────────────────────────────────────────────────

    # 3. Create rounded keys to ensure floating-point decimals match perfectly
    merge_cols = ['Donor', 'Acceptor', 'PLP', 'Enzyme']
    key_cols = []

    RESULTS_DF = DESIGN_DF.copy()

    for col in merge_cols:
        key_name = f'{col}_key'
        key_cols.append(key_name)
        # Convert Excel column to numeric (handles strings) and round
        subset_df[key_name] = pd.to_numeric(subset_df[col], errors='coerce').round(3)
        # Round Python matrix values
        RESULTS_DF[key_name] = RESULTS_DF[col].round(3)

    # 4. Handle the 5 identical center points so they don't multiply during merge
    subset_df['dup_id'] = subset_df.groupby(key_cols).cumcount()
    RESULTS_DF['dup_id'] = RESULTS_DF.groupby(key_cols).cumcount()

    # 5. Smart merge on physical keys! (Ignores Run order completely)
    RESULTS_DF = RESULTS_DF.drop(columns=[RESPONSE_NAME], errors='ignore')
    merged = RESULTS_DF.merge(
        subset_df[key_cols + ['dup_id', RESPONSE_NAME]],
        on=key_cols + ['dup_id'],
        how='left'
    )

    # Clean up temporary keys
    RESULTS_DF = merged.drop(columns=key_cols + ['dup_id'])

    print(f"✓ Successfully mapped yields from '{EXCEL_FILENAME}' based on actual physical factors.\n")

    # ── SEE EXACTLY WHAT IT READ ──────────────────────────────────────────────
    print("── DATA MAPPING CHECK ──────────────────────────────────────────────")
    print("Verify that the Yield was attached to the correct physical parameters:")
    print(RESULTS_DF[['Run', 'Donor', 'Acceptor', 'PLP', 'Enzyme', RESPONSE_NAME]].head(10))
    print("────────────────────────────────────────────────────────────────────\n")

except Exception as e:
    print(f"⚠ Error reading Excel file: {e}")
    RESULTS_DF = DESIGN_DF.copy()
    RESULTS_DF[RESPONSE_NAME] = np.nan

# ── Quick check ───────────────────────────────────────────────────────────────
n_filled = RESULTS_DF[RESPONSE_NAME].notna().sum()
print(f"✓  {n_filled}/{len(RESULTS_DF)} results loaded.")

# Prevent division by zero crash if no variance is found
if n_filled > 0 and RESULTS_DF[RESPONSE_NAME].var() == 0:
    print("\n⚠ WARNING: All mapped yields are identical or zero. Check your Excel file formatting.")

if n_filled < 2**K + 2*K + N_CENTER:
    print(f"⚠  Only {n_filled} results available — analysis sections will use "
          f"placeholder data until you fill all values.\n")

    # ── DEMO MODE ────────────────────────────────────────────────────────────
    rng = np.random.default_rng(42)
    def sim_yield(row):
        x = {fn: row[f'{fn}_coded'] for fn in FACTOR_NAMES}
        y = (85 + 8*x.get('Donor',0) - 12*x.get('Acceptor',0) + 3*x.get('PLP',0)
             - 6*x.get('Enzyme',0) - 5*x.get('Donor',0)**2 - 10*x.get('Acceptor',0)**2)
        if 'Control' in str(row.get('Run_Type','')): return 0.0
        return float(np.clip(y, 0, 120))

    RESULTS_DF[RESPONSE_NAME] = RESULTS_DF.apply(sim_yield, axis=1)
    DEMO_MODE = True
else:
    DEMO_MODE = False

# ── VERIFICATION TABLE ────────────────────────────────────────────────────────
# Run this to double-check your data merge

print("=== DATA MERGE VERIFICATION TABLE ===")
print("Compare the physical values and yields below with your original Excel file.\n")

# Select the essential columns
verify_df = RESULTS_DF[['Run', 'Run_Type', 'Donor', 'Acceptor', 'PLP', 'Enzyme', RESPONSE_NAME]].copy()

# Sort by physical parameters (Donor, then Acceptor) instead of Run number.
# This groups similar runs together, making it MUCH easier to spot-check against your Excel file!
verify_df = verify_df.sort_values(by=['Donor', 'Acceptor', 'PLP', 'Enzyme']).reset_index(drop=True)

# Display the full table in Colab so you can scroll through it
display(verify_df.style.background_gradient(subset=[RESPONSE_NAME], cmap='YlGn'))

# Export it to a CSV so you can download and view it side-by-side in Excel
verify_df.to_csv('Merge_Verification.csv', index=False)
print("\n✓ Exported full table to 'Merge_Verification.csv'")
print("  (Click the Folder icon on the left sidebar in Colab to download it)")

# ══════════════════════════════════════════════════════════════════════════════
# §5 — FIT RSM MODEL & ANOVA
# ══════════════════════════════════════════════════════════════════════════════

# ── Prepare modelling dataset (exclude controls) ──────────────────────────────
ctrl_mask = RESULTS_DF['Run_Type'].str.contains('Control', na=False)
MODEL_DF = RESULTS_DF[~ctrl_mask & RESULTS_DF[RESPONSE_NAME].notna()].copy()

# Short coded variable names for formula (x1, x2, …)
short = {fn: f"x{i+1}" for i, fn in enumerate(FACTOR_NAMES)}
for fn, sn in short.items():
    MODEL_DF[sn] = MODEL_DF[f'{fn}_coded']
SN = list(short.values())

def build_formula(short_names, response='y'):
    """Full second-order formula for statsmodels OLS."""
    lin  = " + ".join(short_names)
    quad = " + ".join([f"I({s}**2)" for s in short_names])
    inter = " + ".join([f"{a}:{b}"
                        for a, b in combinations(short_names, 2)])
    return f"{response} ~ {lin} + {quad} + {inter}"

MODEL_DF['y'] = MODEL_DF[RESPONSE_NAME]
FORMULA = build_formula(SN)

model_fit = ols(FORMULA, data=MODEL_DF).fit()

# ── ANOVA table ───────────────────────────────────────────────────────────────
anova_raw  = sm.stats.anova_lm(model_fit, typ=2)  # Type II SS

# Rename rows to nice labels
term_map = {}
for fn, sn in short.items():
    term_map[sn]             = fn
    term_map[f"I({sn} ** 2)"]  = f"{fn}²"
for (fn1, sn1), (fn2, sn2) in zip(
        list(short.items()), list(short.items())[1:]):
    pass
# Full interaction name map
for (fn_a, sn_a), (fn_b, sn_b) in combinations(short.items(), 2):
    term_map[f"{sn_a}:{sn_b}"] = f"{fn_a} × {fn_b}"

anova_display = anova_raw.copy()
anova_display.index = [term_map.get(i, i) for i in anova_display.index]
anova_display.columns = ['SS', 'df', 'F', 'p-value']
anova_display['Significant'] = anova_display['p-value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else
              ('*' if p < 0.05 else ('.' if p < 0.1 else ''))))

# ── Centre-point pure error → Lack of Fit ────────────────────────────────────
centre_mask = MODEL_DF['Run_Type'] == 'Centre'
centre_vals = MODEL_DF.loc[centre_mask, 'y']
n_centre = len(centre_vals)
pure_error_ss  = float(centre_vals.var(ddof=1) * (n_centre - 1)) if n_centre > 1 else np.nan
pure_error_df  = n_centre - 1 if n_centre > 1 else np.nan
residual_ss    = model_fit.ssr
residual_df    = model_fit.df_resid
lof_ss  = residual_ss - (pure_error_ss if not np.isnan(pure_error_ss) else 0)
lof_df  = residual_df - (pure_error_df if not np.isnan(pure_error_df) else 0)
if not np.isnan(pure_error_ss) and pure_error_df > 0:
    lof_F = (lof_ss / lof_df) / (pure_error_ss / pure_error_df)
    lof_p = 1 - stats.f.cdf(lof_F, lof_df, pure_error_df)
else:
    lof_F = lof_p = np.nan

# ── PRESS / Predicted R² ──────────────────────────────────────────────────────
influence = model_fit.get_influence()
press = np.sum(influence.resid_studentized_internal**2 *
               (1 - influence.hat_matrix_diag)**0 /
               (1 / (1 - influence.hat_matrix_diag))**2
               ) if hasattr(influence, 'hat_matrix_diag') else np.nan
# Simpler: PRESS via leave-one-out residuals = resid / (1 - hii)
hii   = influence.hat_matrix_diag
press = float(np.sum((model_fit.resid / (1 - hii))**2))
ss_tot = float(np.sum((MODEL_DF['y'] - MODEL_DF['y'].mean())**2))
pred_r2 = 1 - press / ss_tot

r2      = model_fit.rsquared
adj_r2  = model_fit.rsquared_adj

# ── Print model summary ───────────────────────────────────────────────────────
print("═" * 65)
print("  RSM MODEL — FULL SECOND-ORDER (QUADRATIC)")
print("═" * 65)
print(f"  R²              {r2:.4f}")
print(f"  Adjusted R²     {adj_r2:.4f}")
print(f"  Predicted R²    {pred_r2:.4f}")
print(f"  PRESS           {press:.2f}")
print(f"  Model F-stat    {model_fit.fvalue:.2f}   p = {model_fit.f_pvalue:.4g}")
if not np.isnan(lof_p):
    lof_ok = "✓ Good fit" if lof_p > 0.05 else "⚠ Possible misfit"
    print(f"  Lack of Fit F   {lof_F:.2f}   p = {lof_p:.4g}   {lof_ok}")
print()

# ── Coefficients table ────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Term':    [term_map.get(p, p) for p in model_fit.params.index],
    'Coeff':   model_fit.params.values,
    'Std Err': model_fit.bse.values,
    't':       model_fit.tvalues.values,
    'p-value': model_fit.pvalues.values,
    '95% CI Low':  model_fit.conf_int()[0].values,
    '95% CI High': model_fit.conf_int()[1].values,
})
coef_df['Sig'] = coef_df['p-value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else
              ('*' if p < 0.05 else ('.' if p < 0.1 else ''))))
coef_df = coef_df.set_index('Term')

print("  COEFFICIENT TABLE")
print("─" * 65)
display(coef_df.style.background_gradient(
    subset=['Coeff'], cmap='RdBu_r', vmin=-15, vmax=15).format({
    'Coeff':'{:.3f}','Std Err':'{:.3f}','t':'{:.2f}',
    'p-value':'{:.4f}','95% CI Low':'{:.3f}','95% CI High':'{:.3f}'}))

print("\n  ANOVA TABLE (Type II SS)")
print("─" * 65)
display(anova_display.style.format({
    'SS':'{:.2f}','df':'{:.0f}','F':'{:.2f}','p-value':'{:.4f}'}).background_gradient(
    subset=['F'], cmap='YlOrRd'))


# ══════════════════════════════════════════════════════════════════════════════
# §6 — DIAGNOSTIC PLOTS
# ══════════════════════════════════════════════════════════════════════════════

fitted   = model_fit.fittedvalues.values
resid    = model_fit.resid.values
std_resid = (resid - resid.mean()) / resid.std()
actual   = MODEL_DF['y'].values

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("RSM Diagnostic Plots", fontsize=14, fontweight='bold',
             color=NAVY, y=1.01)

# ── 1. Predicted vs Actual ────────────────────────────────────────────────────
ax = axes[0, 0]
ax.scatter(actual, fitted, color=TEAL, edgecolors='white', s=70, zorder=3, alpha=0.9)
lo, hi = min(actual.min(), fitted.min()), max(actual.max(), fitted.max())
ax.plot([lo, hi], [lo, hi], color=RED_C, lw=1.5, ls='--', label='Perfect fit')
ax.set_xlabel(f"Actual  {RESPONSE_NAME}"); ax.set_ylabel(f"Predicted  {RESPONSE_NAME}")
ax.set_title("Predicted vs Actual")
ax.legend(fontsize=9)
ax.text(0.05, 0.92, f"R² = {r2:.4f}", transform=ax.transAxes,
        fontsize=9, color=NAVY)

# ── 2. Residuals vs Fitted ────────────────────────────────────────────────────
ax = axes[0, 1]
ax.scatter(fitted, resid, color=NAVY, edgecolors='white', s=70, zorder=3, alpha=0.85)
ax.axhline(0, color=RED_C, lw=1.5, ls='--')
ax.set_xlabel("Fitted values"); ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted")
# Annotate outliers > 2σ
thresh = 2 * resid.std()
for i, (f_, r_) in enumerate(zip(fitted, resid)):
    if abs(r_) > thresh:
        ax.annotate(f" run {MODEL_DF['Run'].iloc[i]}",
                    (f_, r_), fontsize=8, color=RED_C)

# ── 3. Normal Q-Q of residuals ────────────────────────────────────────────────
ax = axes[0, 2]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(resid, dist="norm")
ax.scatter(osm, osr, color=NAVY, edgecolors='white', s=60, zorder=3)
line_x = np.array([osm[0], osm[-1]])
ax.plot(line_x, slope * line_x + intercept, color=RED_C, lw=1.5, ls='--')
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Sample Quantiles")
ax.set_title("Normal Q-Q of Residuals")
ax.text(0.05, 0.92, f"R = {r_qq:.4f}", transform=ax.transAxes,
        fontsize=9, color=NAVY)

# ── 4. Scale-Location ─────────────────────────────────────────────────────────
ax = axes[1, 0]
sqrt_abs_resid = np.sqrt(np.abs(std_resid))
ax.scatter(fitted, sqrt_abs_resid, color=AMBER, edgecolors='white',
           s=70, zorder=3, alpha=0.9)
# Lowess trend
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    trend = lowess(sqrt_abs_resid, fitted, frac=0.6)
    ax.plot(trend[:, 0], trend[:, 1], color=RED_C, lw=1.5)
except Exception:
    pass
ax.set_xlabel("Fitted values"); ax.set_ylabel("√|Standardised residuals|")
ax.set_title("Scale-Location")

# ── 5. Cook's Distance ────────────────────────────────────────────────────────
ax = axes[1, 1]
infl = model_fit.get_influence()
cooks_d = infl.cooks_distance[0]
markerline, stemlines, baseline = ax.stem(
    range(1, len(cooks_d)+1), cooks_d, markerfmt='o', linefmt=f'C0-',
    basefmt='grey')
plt.setp(markerline, color=NAVY, markersize=5)
ax.axhline(4 / len(cooks_d), color=RED_C, ls='--', lw=1.2,
           label=f"Threshold 4/n = {4/len(cooks_d):.3f}")
ax.set_xlabel("Run index"); ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance (Influence)")
ax.legend(fontsize=8)

# ── 6. Box-Cox transformation check ──────────────────────────────────────────
ax = axes[1, 2]
if (MODEL_DF['y'] > 0).all():
    lambdas = np.linspace(-2, 2, 200)
    y_pos = MODEL_DF['y'].values
    log_likes = []
    for lam in lambdas:
        if abs(lam) < 1e-6:
            y_t = np.log(y_pos)
        else:
            y_t = (y_pos**lam - 1) / lam
        n = len(y_t)
        ll = (-n/2) * np.log(np.var(y_t, ddof=1)) + (lam - 1) * np.sum(np.log(y_pos))
        log_likes.append(ll)
    best_lam = lambdas[np.argmax(log_likes)]
    ax.plot(lambdas, log_likes, color=TEAL, lw=2)
    ax.axvline(best_lam, color=RED_C, ls='--', lw=1.5,
               label=f"Best λ = {best_lam:.2f}")
    ax.axvline(1, color=AMBER, ls=':', lw=1.2, label="λ = 1 (no transform)")
    ci_thresh = max(log_likes) - stats.chi2.ppf(0.95, 1) / 2
    ax.axhline(ci_thresh, color=NAVY, ls=':', lw=1, alpha=0.5,
               label="95% CI boundary")
    ax.set_xlabel("λ"); ax.set_ylabel("Log-likelihood")
    ax.set_title("Box-Cox Transformation Check")
    ax.legend(fontsize=8)
    if 0.75 < best_lam < 1.25:
        ax.text(0.05, 0.10, "→ No transformation needed (λ ≈ 1)",
                transform=ax.transAxes, fontsize=9, color=TEAL)
    else:
        ax.text(0.05, 0.10, f"→ Consider λ = {best_lam:.2f} transformation",
                transform=ax.transAxes, fontsize=9, color=RED_C)
else:
    ax.text(0.5, 0.5, "Box-Cox requires y > 0", ha='center', va='center')
    ax.set_title("Box-Cox (not applicable)")

plt.tight_layout()
plt.savefig('diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓  Diagnostic plots saved to diagnostics.png")


# ══════════════════════════════════════════════════════════════════════════════
# §7 — EFFECTS ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Effects Analysis", fontsize=14, fontweight='bold', color=NAVY, y=1.01)
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.38)

# ── A. Pareto chart of standardised effects ───────────────────────────────────
ax_pareto = fig.add_subplot(gs[0, :2])
# Exclude intercept; use |t-values| as standardised effects
coef_no_int = coef_df.drop('Intercept', errors='ignore').copy()
coef_no_int['|t|'] = coef_no_int['t'].abs()
coef_sorted = coef_no_int.sort_values('|t|', ascending=True)

colors_bar = [RED_C if p < 0.05 else (AMBER if p < 0.10 else SLATE)
              for p in coef_sorted['p-value']]
bars = ax_pareto.barh(coef_sorted.index, coef_sorted['|t|'],
                      color=colors_bar, edgecolor='white', height=0.65)
# t-critical at α=0.05
t_crit = stats.t.ppf(0.975, model_fit.df_resid)
ax_pareto.axvline(t_crit, color=RED_C, ls='--', lw=1.5,
                  label=f"t-crit (α=0.05) = {t_crit:.2f}")
ax_pareto.set_xlabel("|t-value|  (standardised effect)")
ax_pareto.set_title("Pareto Chart of Standardised Effects")
ax_pareto.legend(fontsize=9)
for bar, p in zip(bars, coef_sorted['p-value']):
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else
          ('*' if p < 0.05 else ''))
    if sig:
        ax_pareto.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                       sig, va='center', fontsize=9, color=RED_C)

# ── B. Half-normal probability plot of effects ────────────────────────────────
ax_hn = fig.add_subplot(gs[0, 2])
effects = coef_no_int['Coeff'].abs().sort_values()
n_eff = len(effects)
probs = [(i - 0.5) / n_eff for i in range(1, n_eff + 1)]
half_norm_q = [stats.norm.ppf(0.5 + p / 2) for p in probs]
ax_hn.scatter(half_norm_q, effects.values, color=NAVY, edgecolors='white',
              s=60, zorder=3)
for q, eff, name in zip(half_norm_q, effects.values, effects.index):
    if eff > effects.quantile(0.75):
        ax_hn.annotate(f" {name}", (q, eff), fontsize=7.5, color=NAVY)
ax_hn.set_xlabel("Half-normal scores")
ax_hn.set_ylabel("|Effect|")
ax_hn.set_title("Half-normal Plot")

# ── C. Main effects plot ───────────────────────────────────────────────────────
ax_me = fig.add_subplot(gs[1, :2])
coded_levels = [-1, 0, 1]
level_labels = ['Low (−1)', 'Centre (0)', 'High (+1)']
mean_by_level = {}
for fn, sn in short.items():
    means = []
    for lev in coded_levels:
        mask = np.isclose(MODEL_DF[sn], lev, atol=0.05)
        means.append(MODEL_DF.loc[mask, 'y'].mean())
    mean_by_level[fn] = means

grand_mean = MODEL_DF['y'].mean()
x_positions = np.arange(len(coded_levels))
palette = [TEAL, NAVY, AMBER, RED_C, SLATE, "#8E44AD"]
for i, (fn, means) in enumerate(mean_by_level.items()):
    offset = (i - len(mean_by_level)/2) * 0.12
    ax_me.plot(x_positions + offset, means, '-o',
               label=fn, color=palette[i % len(palette)], lw=1.8, ms=7)
ax_me.axhline(grand_mean, color=SLATE, ls=':', lw=1, label='Grand mean')
ax_me.set_xticks(x_positions)
ax_me.set_xticklabels(level_labels)
ax_me.set_ylabel(RESPONSE_NAME)
ax_me.set_title("Main Effects Plot")
ax_me.legend(fontsize=9, ncol=2)

# ── D. Interaction heatmap (2-way interaction coefficients) ───────────────────
ax_ih = fig.add_subplot(gs[1, 2])
inter_coefs = np.zeros((K, K))
np.fill_diagonal(inter_coefs, np.nan)
for (fi, si), (fj, sj) in combinations(enumerate(FACTOR_NAMES), 2):
    key = f"{short[si]} × {short[sj]}"
    key2 = f"{FACTOR_NAMES[fi]} × {FACTOR_NAMES[fj]}"
    val = coef_df.loc[key2, 'Coeff'] if key2 in coef_df.index else 0.0
    inter_coefs[fi, fj] = val
    inter_coefs[fj, fi] = val
mask_nan = np.eye(K, dtype=bool)
im = ax_ih.imshow(inter_coefs, cmap='RdBu_r', vmin=-15, vmax=15, aspect='auto')
ax_ih.set_xticks(range(K)); ax_ih.set_yticks(range(K))
ax_ih.set_xticklabels(FACTOR_NAMES, fontsize=8, rotation=30, ha='right')
ax_ih.set_yticklabels(FACTOR_NAMES, fontsize=8)
ax_ih.set_title("2-Way Interaction\nCoefficients")
for i in range(K):
    for j in range(K):
        if i != j:
            ax_ih.text(j, i, f"{inter_coefs[i,j]:.2f}",
                       ha='center', va='center', fontsize=8,
                       color='white' if abs(inter_coefs[i,j]) > 7 else 'black')
plt.colorbar(im, ax=ax_ih, shrink=0.8)

plt.savefig('effects_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓  Effects plots saved to effects_analysis.png")


# ══════════════════════════════════════════════════════════════════════════════
# §8 — RESPONSE SURFACE PLOTS
#   One 3D surface + contour pair for each factor combination
#   All other factors held at their centre point
# ══════════════════════════════════════════════════════════════════════════════

N_GRID   = 60
FACTOR_PAIRS = list(combinations(FACTOR_NAMES, 2))
N_PAIRS  = len(FACTOR_PAIRS)
NCOLS    = min(3, N_PAIRS)
NROWS    = int(np.ceil(N_PAIRS / NCOLS))

# ── Build prediction function from fitted model ───────────────────────────────
def predict_grid(fn_x, fn_y, n_grid=N_GRID, fixed_coded=None):
    """
    Predict response over a 2D grid of fn_x, fn_y (coded −1…+1)
    All other factors fixed at coded 0 (or values in fixed_coded dict).
    Returns xi_coded, yi_coded, z_pred (all real-unit if decode=True)
    """
    xi = np.linspace(-1, 1, n_grid)
    yi = np.linspace(-1, 1, n_grid)
    XI, YI = np.meshgrid(xi, yi)

    rows = []
    for xv, yv in zip(XI.ravel(), YI.ravel()):
        row = {'y': 0}
        for fn, sn in short.items():
            if fn == fn_x:   row[sn] = xv
            elif fn == fn_y: row[sn] = yv
            else:
                row[sn] = (fixed_coded or {}).get(fn, 0.0)
        rows.append(row)
    pred_df = pd.DataFrame(rows)
    ZI = model_fit.predict(pred_df).values.reshape(XI.shape)

    # Convert coded → real for axis labels
    XI_real = np.array([[decode(v, fn_x) for v in row] for row in XI])
    YI_real = np.array([[decode(v, fn_y) for v in row] for row in YI])
    return XI_real, YI_real, ZI

'''# ── 3D Response surfaces ──────────────────────────────────────────────────────
# Style: clean white pane walls, gray grid, viridis surface, black data points
# matching publication-quality RSM plots

def style_3d_axes(ax):
    """Apply clean white-wall style to a 3D axes object."""
    # White pane backgrounds with slight transparency
    ax.xaxis.pane.fill = True
    ax.yaxis.pane.fill = True
    ax.zaxis.pane.fill = True
    ax.xaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.85))
    ax.yaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.85))
    ax.zaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.85))
    # Thin gray pane edges
    ax.xaxis.pane.set_edgecolor('0.75')
    ax.yaxis.pane.set_edgecolor('0.75')
    ax.zaxis.pane.set_edgecolor('0.75')
    ax.xaxis.pane.set_linewidth(0.5)
    ax.yaxis.pane.set_linewidth(0.5)
    ax.zaxis.pane.set_linewidth(0.5)
    # Gray grid lines on all panes
    ax.xaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.5})
    ax.yaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.5})
    ax.zaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.5})
    ax.grid(True)
    # Thin axis lines
    ax.xaxis.line.set_color('0.5'); ax.xaxis.line.set_linewidth(0.8)
    ax.yaxis.line.set_color('0.5'); ax.yaxis.line.set_linewidth(0.8)
    ax.zaxis.line.set_color('0.5'); ax.zaxis.line.set_linewidth(0.8)
    # Tick styling
    ax.tick_params(axis='both', labelsize=8, pad=2, colors='0.3')
    ax.tick_params(axis='z',    labelsize=8, pad=2, colors='0.3')

fig3d = plt.figure(figsize=(6.5 * NCOLS, 5.5 * NROWS),
                   facecolor='white')
fig3d.suptitle("Response Surface Plots (3D)  —  other factors held at centre",
               fontsize=13, fontweight='bold', color='0.15', y=1.01)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):
    ax = fig3d.add_subplot(NROWS, NCOLS, idx + 1, projection='3d')
    ax.set_facecolor('white')

    XI, YI, ZI = predict_grid(fn_x, fn_y)

    # Surface: viridis with light wireframe to show curvature
    surf = ax.plot_surface(
        XI, YI, ZI,
        cmap=CMAP_RS,
        alpha=0.88,
        linewidth=0.2,
        edgecolor='0.6',
        antialiased=True,
        rcount=40, ccount=40,
    )

    # Black data points raised slightly above surface for visibility
    x_act = [decode(MODEL_DF[short[fn_x]].iloc[i], fn_x)
              for i in range(len(MODEL_DF))]
    y_act = [decode(MODEL_DF[short[fn_y]].iloc[i], fn_y)
              for i in range(len(MODEL_DF))]
    z_act = MODEL_DF['y'].values
    ax.scatter(x_act, y_act, z_act,
               color='black', edgecolors='black',
               s=28, zorder=10, alpha=0.85,
               depthshade=True, label='Measured')

    # Axes labels
    fx = FACTORS[fn_x]; fy = FACTORS[fn_y]
    ax.set_xlabel(f"{fn_x} ({fx['unit']})", fontsize=11, labelpad=6, color='0.2')
    ax.set_ylabel(f"{fn_y} ({fy['unit']})", fontsize=11, labelpad=6, color='0.2')
    ax.set_zlabel(RESPONSE_NAME,             fontsize=11, labelpad=6, color='0.2')
    #ax.set_title(f"{fn_x}  ×  {fn_y}", fontsize=11, fontweight='bold',
    #             color='0.15', pad=10)

    # Apply clean wall style
    style_3d_axes(ax)

    # Viewing angle (matches reference image perspective)
    ax.view_init(elev=22, azim=-55)

    # Colourbar
    cb = fig3d.colorbar(surf, ax=ax, shrink=0.45, pad=0.08,
                        orientation='vertical')
    cb.ax.tick_params(labelsize=9)
    cb.set_label(RESPONSE_NAME, fontsize=10, color='0.3')
    cb.outline.set_linewidth(0.5)

# Legend on first subplot
handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor='black', markersize=6,
                       label='Measured')]
fig3d.legend(handles=handles, loc='lower center',
             ncol=1, fontsize=10, framealpha=0.9,
             edgecolor='0.8', bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('response_surfaces_3d.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✓  3D surfaces saved to response_surfaces_3d.png")'''

# ── 3D Response surfaces ──────────────────────────────────────────────────────
# Publication-quality style with minimal changes from the original

def style_3d_axes(ax):
    """Apply clean white-wall style to a 3D axes object."""

    # White pane backgrounds
    ax.xaxis.pane.fill = True
    ax.yaxis.pane.fill = True
    ax.zaxis.pane.fill = True

    ax.xaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.90))
    ax.yaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.90))
    ax.zaxis.pane.set_facecolor((0.96, 0.96, 0.96, 0.90))

    # Thin gray pane edges
    ax.xaxis.pane.set_edgecolor('0.75')
    ax.yaxis.pane.set_edgecolor('0.75')
    ax.zaxis.pane.set_edgecolor('0.75')

    ax.xaxis.pane.set_linewidth(0.6)
    ax.yaxis.pane.set_linewidth(0.6)
    ax.zaxis.pane.set_linewidth(0.6)

    # Gray grid lines
    ax.xaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.6})
    ax.yaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.6})
    ax.zaxis._axinfo['grid'].update({'color': '0.80', 'linewidth': 0.6})

    ax.grid(True)

    # Slightly thicker axis lines
    ax.xaxis.line.set_color('0.45')
    ax.yaxis.line.set_color('0.45')
    ax.zaxis.line.set_color('0.45')

    ax.xaxis.line.set_linewidth(1.0)
    ax.yaxis.line.set_linewidth(1.0)
    ax.zaxis.line.set_linewidth(1.0)

    # Larger tick labels
    ax.tick_params(axis='both',
                   labelsize=11,
                   pad=2,
                   colors='0.25')

    ax.tick_params(axis='z',
                   labelsize=11,
                   pad=2,
                   colors='0.25')


# ───────────────────────────────────────────────────────────────

fig3d = plt.figure(
    figsize=(7.5 * NCOLS, 6.5 * NROWS),
    facecolor='white'
)

fig3d.suptitle(
    "Response Surface Plots (3D) — other factors held at centre",
    fontsize=15,
    fontweight='bold',
    color='0.15',
    y=1.01
)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):

    ax = fig3d.add_subplot(
        NROWS,
        NCOLS,
        idx + 1,
        projection='3d'
    )

    ax.set_facecolor('white')

    XI, YI, ZI = predict_grid(fn_x, fn_y)

    # Response surface
    surf = ax.plot_surface(
        XI,
        YI,
        ZI,
        cmap=CMAP_RS,
        alpha=0.90,
        linewidth=0.2,
        edgecolor='0.6',
        antialiased=True,
        rcount=80,
        ccount=80,
    )

    # Experimental points
    x_act = [
        decode(MODEL_DF[short[fn_x]].iloc[i], fn_x)
        for i in range(len(MODEL_DF))
    ]

    y_act = [
        decode(MODEL_DF[short[fn_y]].iloc[i], fn_y)
        for i in range(len(MODEL_DF))
    ]

    z_act = MODEL_DF['y'].values

    ax.scatter(
        x_act,
        y_act,
        z_act,
        color='black',
        edgecolors='white',
        linewidth=0.4,
        s=40,
        alpha=0.9,
        depthshade=True,
        zorder=10,
        label='Measured'
    )

    # Axis labels
    fx = FACTORS[fn_x]
    fy = FACTORS[fn_y]

    ax.set_xlabel(
        f"{fn_x} ({fx['unit']})",
        fontsize=13,
        labelpad=2,
        color='0.2'
    )

    ax.set_ylabel(
        f"{fn_y} ({fy['unit']})",
        fontsize=13,
        labelpad=2,
        color='0.2'
    )

    ax.set_zlabel(
        RESPONSE_NAME,
        fontsize=13,
        labelpad=3,
        color='0.2'
    )

    # Apply styling
    style_3d_axes(ax)

    # Slightly improved viewing angle
    ax.view_init(elev=26, azim=-58)

    # Colorbar
    cb = fig3d.colorbar(
        surf,
        ax=ax,
        shrink=0.55,
        pad=0.05,
        orientation='vertical'
    )

    cb.ax.tick_params(labelsize=11)
    cb.outline.set_linewidth(0.6)

# Legend
handles = [
    plt.Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='black',
        markeredgecolor='white',
        markeredgewidth=0.4,
        markersize=7,
        label='Measured'
    )
]

fig3d.legend(
    handles=handles,
    loc='lower center',
    ncol=1,
    fontsize=11,
    framealpha=0.9,
    edgecolor='0.8',
    bbox_to_anchor=(0.5, -0.02)
)

plt.tight_layout(rect=[0, 0.03, 1, 1])

plt.savefig(
    'response_surfaces_3d.png',
    dpi=600,          # publication quality
    bbox_inches='tight',
    facecolor='white'
)

plt.show()

print("✓ 3D surfaces saved to response_surfaces_3d.png")

# ── Contour plots ─────────────────────────────────────────────────────────────
fig_ct = plt.figure(figsize=(6 * NCOLS, 5 * NROWS), facecolor='white')
fig_ct.suptitle(
    "Response Surface Contour Plots  —  other factors at centre",
    fontsize=13,
    fontweight='bold',
    color=NAVY,
    y=1.01
)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):
    ax = fig_ct.add_subplot(NROWS, NCOLS, idx + 1)

    # White background
    ax.set_facecolor('white')

    XI, YI, ZI = predict_grid(fn_x, fn_y)

    cp = ax.contourf(
        XI, YI, ZI,
        levels=20,
        cmap=CMAP_RS
    )

    cs = ax.contour(
        XI, YI, ZI,
        levels=10,
        colors='black',
        linewidths=0.6,
        alpha=0.6
    )

    ax.clabel(
        cs,
        fmt='%.0f',
        fontsize=9,
        colors='black'
    )

    cbar = plt.colorbar(cp, ax=ax)
    cbar.set_label(RESPONSE_NAME)
    cbar.ax.set_facecolor('white')

    # Scatter actual data
    x_act = [
        decode(MODEL_DF[short[fn_x]].iloc[i], fn_x)
        for i in range(len(MODEL_DF))
    ]
    y_act = [
        decode(MODEL_DF[short[fn_y]].iloc[i], fn_y)
        for i in range(len(MODEL_DF))
    ]

    ax.scatter(
        x_act,
        y_act,
        color='white',
        edgecolors='black',
        s=35,
        zorder=5,
        linewidths=0.8
    )

    fx = FACTORS[fn_x]
    fy = FACTORS[fn_y]

    ax.set_xlabel(f"{fn_x} ({fx['unit']})")
    ax.set_ylabel(f"{fn_y} ({fy['unit']})")
    ax.set_title(f"{fn_x} × {fn_y}", fontsize=10)

    # Factor range guides
    ax.axvline(fx['low'],  color='black', ls=':', lw=0.8, alpha=0.5)
    ax.axvline(fx['high'], color='black', ls=':', lw=0.8, alpha=0.5)
    ax.axhline(fy['low'],  color='black', ls=':', lw=0.8, alpha=0.5)
    ax.axhline(fy['high'], color='black', ls=':', lw=0.8, alpha=0.5)

    # White panel with black border
    for spine in ax.spines.values():
        spine.set_color('black')

plt.tight_layout()

plt.savefig(
    'contour_plots.png',
    dpi=130,
    bbox_inches='tight',
    facecolor='white',
    edgecolor='white'
)

plt.show()

print("✓  Contour plots saved to contour_plots.png")



# ── Global Font Optimization for Publication ───────────────────────────────────
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.edgecolor': 'white'
})

# ── Perturbation plot ─────────────────────────────────────────────────────────
fig_pert, ax_pert = plt.subplots(figsize=(10, 5), facecolor='white')
ax_pert.set_facecolor('white')

xi_range = np.linspace(-1, 1, 200)
palette_p = [TEAL, NAVY, AMBER, RED_C, SLATE, "#8E44AD"]

for i, (fn, sn) in enumerate(short.items()):
    y_vals = []
    for xi in xi_range:
        row = {sn2: 0.0 for sn2 in SN}
        row[sn] = xi
        row['y'] = 0
        y_pred = model_fit.predict(pd.DataFrame([row]))[0]
        y_vals.append(y_pred)

    ax_pert.plot(
        xi_range,
        y_vals,
        label=fn,
        color=palette_p[i % len(palette_p)],
        lw=2
    )

# Centre line
ax_pert.axvline(0, color='black', ls=':', lw=1, alpha=0.5)

# Labels and title
ax_pert.set_xlabel("Coded factor value (all others at centre = 0)")
ax_pert.set_ylabel(f"Predicted {RESPONSE_NAME}")
ax_pert.set_title("Perturbation Plot — effect of each factor individually from centre")

# Ticks
ax_pert.set_xticks([-1, -0.5, 0, 0.5, 1])
ax_pert.set_xticklabels([
    'Low\n(−1)',
    '−0.5',
    'Centre\n(0)',
    '+0.5',
    'High\n(+1)'
])

# White background with black axes
for spine in ax_pert.spines.values():
    spine.set_color('black')

ax_pert.tick_params(axis='both', colors='black')

# Optional light grid
ax_pert.grid(False)

# Legend
ax_pert.legend(fontsize=10, frameon=False)

plt.tight_layout()

plt.savefig(
    'perturbation.png',
    dpi=150,
    bbox_inches='tight',
    facecolor='white',
    edgecolor='white'
)

plt.show()

print("✓  Perturbation plot saved to perturbation.png")

# ══════════════════════════════════════════════════════════════════════════════
# §9 — OPTIMISATION
#   Finds the combination of factor levels that maximises (or achieves a
#   target) the predicted response within the coded design space [−1, +1]
# ══════════════════════════════════════════════════════════════════════════════

OPTIMISE_FOR = 'maximise'   # 'maximise' | 'minimise' | 'target'
TARGET_VALUE = 95            # only used if OPTIMISE_FOR == 'target'

def predict_from_coded(coded_array):
    row = {'y': 0}
    for i, sn in enumerate(SN):
        row[sn] = coded_array[i]
    return float(model_fit.predict(pd.DataFrame([row]))[0])

if OPTIMISE_FOR == 'maximise':
    obj = lambda x: -predict_from_coded(x)
elif OPTIMISE_FOR == 'minimise':
    obj = lambda x:  predict_from_coded(x)
else:
    obj = lambda x: (predict_from_coded(x) - TARGET_VALUE) ** 2

bounds = [(-1, 1)] * K
# Global optimisation with differential evolution, then refine
de_result = differential_evolution(obj, bounds, seed=42,
                                    maxiter=5000, tol=1e-8, polish=True)
opt_coded = de_result.x
opt_pred  = predict_from_coded(opt_coded)

print("\n" + "═"*65)
print("  OPTIMISATION RESULTS")
print("═"*65)
print(f"  Objective         : {OPTIMISE_FOR.upper()}"
      + (f" (target = {TARGET_VALUE})" if OPTIMISE_FOR == 'target' else ""))
print(f"  Predicted optimum : {opt_pred:.2f}  {RESPONSE_NAME}")
print()
print(f"  {'Factor':<16}  {'Coded':>8}  {'Real value':>14}  {'Unit'}")
print("  " + "─"*55)
opt_real = {}
for i, fn in enumerate(FACTOR_NAMES):
    rv = decode(opt_coded[i], fn)
    opt_real[fn] = rv
    print(f"  {fn:<16}  {opt_coded[i]:>+8.3f}  {rv:>14.4g}  {FACTORS[fn]['unit']}")

print()

# ── Visualise optimum on each contour pair ────────────────────────────────────
fig_opt = plt.figure(figsize=(6 * NCOLS, 5 * NROWS))
fig_opt.suptitle(f"Optimum location on contour plots  (★ = predicted optimum)",
                 fontsize=13, fontweight='bold', color=NAVY, y=1.01)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):
    ax = fig_opt.add_subplot(NROWS, NCOLS, idx + 1)
    XI, YI, ZI = predict_grid(fn_x, fn_y)
    cp = ax.contourf(XI, YI, ZI, levels=20, cmap=CMAP_RS, alpha=0.88)
    cs = ax.contour(XI, YI, ZI, levels=10, colors='white', linewidths=0.4, alpha=0.5)
    ax.clabel(cs, fmt='%.0f', fontsize=7, colors='white')
    plt.colorbar(cp, ax=ax, label=RESPONSE_NAME)
    ax.scatter([opt_real[fn_x]], [opt_real[fn_y]],
               marker='*', s=280, color='gold', edgecolors='black',
               linewidths=0.8, zorder=10, label='Optimum')
    ax.legend(fontsize=9)
    fx = FACTORS[fn_x]; fy = FACTORS[fn_y]
    ax.set_xlabel(f"{fn_x}  ({fx['unit']})")
    ax.set_ylabel(f"{fn_y}  ({fy['unit']})")
    ax.set_title(f"{fn_x} × {fn_y}")

plt.tight_layout()
plt.savefig('optimum_location.png', dpi=130, bbox_inches='tight')
plt.show()
print("✓  Optimum plots saved to optimum_location.png")


# ══════════════════════════════════════════════════════════════════════════════
# §10 — SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════

def sig_flag(p):
    return '***' if p < 0.001 else ('**' if p < 0.01 else
           ('*' if p < 0.05 else ('.' if p < 0.1 else 'ns')))

sig_terms = coef_df[coef_df['p-value'] < 0.05].drop('Intercept', errors='ignore')
sig_names = sig_terms.index.tolist()

report_lines = [
    "═"*65,
    "  RSM ANALYSIS SUMMARY REPORT",
    "═"*65,
    f"  Response variable   : {RESPONSE_NAME}",
    f"  Design              : Face-centred CCD, {K} factors, {len(MODEL_DF)} modelling points",
    "",
    "  MODEL FIT",
    f"  R²           = {r2:.4f}",
    f"  Adjusted R²  = {adj_r2:.4f}",
    f"  Predicted R² = {pred_r2:.4f}",
]
if not np.isnan(lof_p):
    report_lines += [
        f"  Lack of Fit  F = {lof_F:.2f},  p = {lof_p:.4g}  "
        f"({'adequate fit' if lof_p > 0.05 else 'potential misfit — check'})"
    ]
report_lines += [
    "",
    "  SIGNIFICANT TERMS (p < 0.05)",
]
if sig_names:
    for nm in sig_names:
        coeff = coef_df.loc[nm, 'Coeff']
        pval  = coef_df.loc[nm, 'p-value']
        direction = "↑" if coeff > 0 else "↓"
        report_lines.append(
            f"  {nm:<26}  coeff = {coeff:+.3f}   p = {pval:.4f} {sig_flag(pval)} {direction}")
else:
    report_lines.append("  No terms significant at α = 0.05 — check your results or model.")

report_lines += [
    "",
    "  OPTIMAL CONDITIONS",
    f"  Predicted {RESPONSE_NAME} = {opt_pred:.2f}",
]
for fn in FACTOR_NAMES:
    report_lines.append(
        f"  {fn:<16} = {opt_real[fn]:.4g} {FACTORS[fn]['unit']}"
        f"  (coded: {encode(opt_real[fn], fn):+.3f})")

report_lines += [
    "",
    "  RECOMMENDATION",
]
if adj_r2 > 0.85:
    report_lines.append("  ✓ Model quality is good (Adj-R² > 0.85). Proceed to verification.")
elif adj_r2 > 0.70:
    report_lines.append("  ⚠ Moderate model quality. Check residuals; consider augmentation.")
else:
    report_lines.append("  ✗ Poor model fit. Review factor ranges and check for outliers.")

if not np.isnan(lof_p) and lof_p < 0.05:
    report_lines.append("  ⚠ Significant lack of fit — model may need additional terms or "
                        "range adjustment.")

report_lines.append("")
report_lines.append("  Generated by RSM_CCD_Optimisation.py")
report_lines.append("═"*65)

print("\n".join(report_lines))

# Save to file
with open("RSM_summary_report.txt", "w") as fh:
    fh.write("\n".join(report_lines))

print("\n✓  Summary saved to RSM_summary_report.txt")
print("✓  All plots saved as PNG files in the working directory.")

try:
    for fname in ['CCD_design_matrix.csv','RSM_summary_report.txt',
                  'diagnostics.png','effects_analysis.png',
                  'response_surfaces_3d.png','contour_plots.png',
                  'perturbation.png','optimum_location.png']:
        if os.path.exists(fname):
            display(FileLink(fname, result_html_prefix=f"⬇  {fname}: "))
except Exception:
    pass

## Titer optimization

The next cell calculates product titer as:

**Titer (mM) = predicted yield (%) × acceptor concentration (mM) / 100**

Run this cell only after the core cell has successfully fitted `model_fit` and created `MODEL_DF`. The factor must be named exactly **`Acceptor`** because the code searches for that label. If yield is not defined relative to acceptor concentration, revise the titer definition before reuse.

This cell reports the titer optimum and generates titer response-surface and contour plots. Optimization uses differential evolution with a fixed random seed of 42.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §11 — TITER OPTIMISATION & PLOTS (ADD-ON)
#   Calculates Titer based on the modeled Yield and physical Acceptor factor.
#   Titer = Yield (%) × Acceptor (mM) / 100
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*65)
print("  TITER OPTIMISATION RESULTS")
print("═"*65)

# Calculate actual Titer in MODEL_DF for scatter plot overlays
# (Yield is in %, Acceptor is in its physical unit e.g., mM)
if 'Acceptor' in FACTOR_NAMES:
    MODEL_DF['Titer'] = (MODEL_DF['y'] / 100.0) * MODEL_DF['Acceptor']
else:
    print("⚠ WARNING: 'Acceptor' factor not found. Titer calculation requires an 'Acceptor' factor.")

def get_titer(coded_array):
    """Predicts Titer dynamically using the fitted Yield model * Acceptor."""
    # 1. Predict Yield from the existing OLS model
    row = {'y': 0}
    for i, sn in enumerate(SN):
        row[sn] = coded_array[i]
    yield_pred = float(model_fit.predict(pd.DataFrame([row]))[0])

    # 2. Extract real Acceptor concentration
    acc_idx = FACTOR_NAMES.index('Acceptor')
    acc_real = decode(coded_array[acc_idx], 'Acceptor')

    # 3. Calculate Titer
    titer = (yield_pred / 100.0) * acc_real
    return titer

# Optimise for Maximum Titer using Global Differential Evolution
obj_titer = lambda x: -get_titer(x) # Negative for minimisation engine
bounds = [(-1, 1)] * K
titer_de = differential_evolution(obj_titer, bounds, seed=42, maxiter=5000, tol=1e-8, polish=True)

opt_titer_coded = titer_de.x
opt_titer_pred  = get_titer(opt_titer_coded)

print(f"  Objective         : MAXIMISE TITER")
print(f"  Predicted optimum : {opt_titer_pred:.2f} (Titer units)")
print()
print(f"  {'Factor':<16}  {'Coded':>8}  {'Real value':>14}  {'Unit'}")
print("  " + "─"*55)
opt_titer_real = {}
for i, fn in enumerate(FACTOR_NAMES):
    rv = decode(opt_titer_coded[i], fn)
    opt_titer_real[fn] = rv
    print(f"  {fn:<16}  {opt_titer_coded[i]:>+8.3f}  {rv:>14.4g}  {FACTORS[fn]['unit']}")
print()


# ── Prediction Grid Generator for Titer ───────────────────────────────────────
def predict_titer_grid(fn_x, fn_y, n_grid=N_GRID):
    xi = np.linspace(-1, 1, n_grid)
    yi = np.linspace(-1, 1, n_grid)
    XI, YI = np.meshgrid(xi, yi)

    rows = []
    for xv, yv in zip(XI.ravel(), YI.ravel()):
        row = {'y': 0}
        for fn, sn in short.items():
            if fn == fn_x:   row[sn] = xv
            elif fn == fn_y: row[sn] = yv
            else:            row[sn] = 0.0 # Held at Centre
        rows.append(row)

    pred_df = pd.DataFrame(rows)
    # Predict array of Yields
    yield_pred = model_fit.predict(pred_df).values

    # Calculate real acceptor for every point on the grid
    acc_sn = short['Acceptor']
    acc_coded_array = pred_df[acc_sn].values
    acc_real = np.array([decode(v, 'Acceptor') for v in acc_coded_array])

    # Calculate grid Titer values
    ZI_TITER = (yield_pred / 100.0) * acc_real
    ZI_TITER = ZI_TITER.reshape(XI.shape)

    # Decode grid axes to real units for plotting
    XI_real = np.array([[decode(v, fn_x) for v in row] for row in XI])
    YI_real = np.array([[decode(v, fn_y) for v in row] for row in YI])

    return XI_real, YI_real, ZI_TITER


# ── 3D Response surfaces for TITER ────────────────────────────────────────────
fig3d_t = plt.figure(figsize=(7.5 * NCOLS, 6.5 * NROWS), facecolor='#FFFFFF', edgecolor='#FFFFFF')
fig3d_t.suptitle("Titer Response Surfaces (3D) — other factors held at centre",
                 fontsize=18, fontweight='bold', color='0.15', y=1.02)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):
    ax = fig3d_t.add_subplot(NROWS, NCOLS, idx + 1, projection='3d')
    ax.set_facecolor('#FFFFFF')

    XI, YI, ZI_TITER = predict_titer_grid(fn_x, fn_y)

    # Plot surface
    surf = ax.plot_surface(XI, YI, ZI_TITER, cmap=CMAP_RS, alpha=0.90, linewidth=0.2,
                           edgecolor='0.6', antialiased=True, rcount=60, ccount=60)

    # Plot measured data points
    x_act = [decode(MODEL_DF[short[fn_x]].iloc[i], fn_x) for i in range(len(MODEL_DF))]
    y_act = [decode(MODEL_DF[short[fn_y]].iloc[i], fn_y) for i in range(len(MODEL_DF))]
    z_act = MODEL_DF['Titer'].values
    ax.scatter(x_act, y_act, z_act, color='black', edgecolors='white', linewidth=0.4,
               s=40, alpha=0.9, depthshade=True, zorder=10, label='Measured')

    # Styling
    fx = FACTORS[fn_x]; fy = FACTORS[fn_y]
    ax.set_xlabel(f"{fn_x} ({fx['unit']})", fontsize=14, labelpad=8, color='0.2')
    ax.set_ylabel(f"{fn_y} ({fy['unit']})", fontsize=14, labelpad=8, color='0.2')
    ax.set_zlabel("Titer", fontsize=14, labelpad=8, color='0.2')

    style_3d_axes(ax)
    ax.view_init(elev=26, azim=-58)

    cb = fig3d_t.colorbar(surf, ax=ax, shrink=0.55, pad=0.08, orientation='vertical')
    cb.ax.tick_params(labelsize=11)
    cb.set_label("Titer", fontsize=14, labelpad=10, color='0.2')
    cb.outline.set_linewidth(0.6)

plt.tight_layout(pad=3.0, rect=[0, 0.03, 1, 0.98])
plt.savefig('titer_surfaces_3d.png', dpi=300, bbox_inches='tight', facecolor='#FFFFFF', transparent=False)
plt.show()
print("✓  Titer 3D surfaces saved to titer_surfaces_3d.png")


# ── Contour plots for TITER ───────────────────────────────────────────────────
fig_ct_t = plt.figure(figsize=(12 * NCOLS, 9.5 * NROWS), facecolor='#FFFFFF', edgecolor='#FFFFFF', layout='constrained')
fig_ct_t.suptitle("Titer Response Contour Plots — optimum marked with ★",
                  fontsize=26, fontweight='bold', color=NAVY)

for idx, (fn_x, fn_y) in enumerate(FACTOR_PAIRS):
    ax = fig_ct_t.add_subplot(NROWS, NCOLS, idx + 1)
    ax.set_facecolor('#FFFFFF')

    XI, YI, ZI_TITER = predict_titer_grid(fn_x, fn_y)

    cp = ax.contourf(XI, YI, ZI_TITER, levels=20, cmap=CMAP_RS, alpha=0.9)
    cs = ax.contour(XI, YI, ZI_TITER, levels=10, colors='white', linewidths=0.5, alpha=0.6)
    ax.clabel(cs, fmt='%.1f', fontsize=16, colors='white')

    cb = plt.colorbar(cp, ax=ax, pad=0.02)
    cb.set_label("Titer", fontsize=20, labelpad=15)
    cb.ax.tick_params(labelsize=16)

    # Scatter measured points
    x_act = [decode(MODEL_DF[short[fn_x]].iloc[i], fn_x) for i in range(len(MODEL_DF))]
    y_act = [decode(MODEL_DF[short[fn_y]].iloc[i], fn_y) for i in range(len(MODEL_DF))]
    ax.scatter(x_act, y_act, color='white', edgecolors='black', s=80, zorder=5, linewidths=1.5)

    # Plot the predicted optimum for Titer as a Gold Star
    ax.scatter([opt_titer_real[fn_x]], [opt_titer_real[fn_y]],
               marker='*', s=600, color='gold', edgecolors='black',
               linewidths=1.5, zorder=10, label='Optimum Titer')

    # Styling
    fx = FACTORS[fn_x]; fy = FACTORS[fn_y]
    ax.set_xlabel(f"{fn_x}  ({fx['unit']})", fontsize=20, labelpad=12)
    ax.set_ylabel(f"{fn_y}  ({fy['unit']})", fontsize=20, labelpad=12)
    ax.set_title(f"{fn_x} × {fn_y}", fontsize=18, pad=12)
    ax.tick_params(axis='both', labelsize=18, pad=6)

    ax.axvline(fx['low'],  color='white', ls=':', lw=1.2, alpha=0.7)
    ax.axvline(fx['high'], color='white', ls=':', lw=1.2, alpha=0.7)
    ax.axhline(fy['low'],  color='white', ls=':', lw=1.2, alpha=0.7)
    ax.axhline(fy['high'], color='white', ls=':', lw=1.2, alpha=0.7)

plt.savefig('titer_contour_plots.png', dpi=300, bbox_inches='tight', facecolor='#FFFFFF', edgecolor='#FFFFFF', transparent=False)
plt.show()
print("✓  Titer contour plots saved to titer_contour_plots.png (Pure White, Full-Size Plots)")

## Multi-objective economic optimization

The next cell balances predicted yield and titer against donor, PLP, and enzyme usage using a Derringer–Suich desirability function.

### Parameters you may change

- **`TITER_BASIS`**: factor against which yield is defined; currently `Acceptor`.
- **`IMPORTANCE`**: relative weights assigned to Yield, Titer, IPA, PLP, and Enzyme.
- **`MIN_YIELD_FLOOR`**: minimum acceptable predicted yield; currently 85%. Set to `None` to remove this constraint.

Increase the Yield or Titer weights when maintaining performance is more important. Increase the IPA, PLP, or Enzyme weights when minimizing reagent use is more important. Any changes should be documented because they alter the selected economic optimum.

Run this cell after both the core model and titer-optimization cell. It compares the yield-only, titer-only, and economic optima and exports an economic-optimization figure and report.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §11 — ECONOMIC OPTIMISATION (MULTI-OBJECTIVE DESIRABILITY FUNCTION)
#   §9 finds the single best point for Yield alone. That point (7 eq IPA,
#   5 mM acceptor, 1 mM PLP, 0.72 mg/mL enzyme) pushes IPA and PLP to their
#   upper bounds even though the ANOVA/effects analysis shows both have a
#   small effect on yield — i.e. you pay for reagent you barely need.
#
#   This section instead finds the point that maximises Yield AND Titer
#   (product concentration) while simultaneously minimising the three cost
#   drivers — IPA (Donor eq), PLP, and Enzyme — using the classic
#   Derringer–Suich desirability function (the same approach used by
#   Design-Expert / JMP for "numerical optimisation with multiple goals").
#
#   Titer is estimated as Yield(%) × Acceptor (mM), i.e. product formed per
#   litre, since Acceptor is the substrate the yield is calculated against.
#   ⚠ If your Yield is instead defined relative to Donor, change TITER_BASIS
#   below to 'Donor'.
#
#   NOTE ON "IPA": the Donor factor is defined in the model as *equivalents*
#   relative to Acceptor. Minimising Donor-eq minimises the excess amine
#   donor used per mole of acceptor — the standard economic lever. If you'd
#   rather minimise the *absolute* mass of IPA consumed (Donor_eq × Acceptor),
#   set IPA_BASIS = 'absolute' below.
# ══════════════════════════════════════════════════════════════════════════════

TITER_BASIS = 'Acceptor'   # response the % yield is calculated against
IPA_BASIS   = 'equivalents'  # 'equivalents' (Donor eq) or 'absolute' (Donor eq × Acceptor, mM)

# ── Importance weights (how much each goal matters, 1 = normal, higher = more) ─
# Raise YIELD/TITER further if you refuse to trade away conversion; raise the
# reagent weights further if cost matters more than a few % yield.
IMPORTANCE = {
    'Yield':   3,
    'Titer':   4,
    'IPA':     1,   # Donor eq
    'PLP':     1,
    'Enzyme':  2,
}

# Optional hard floor — refuse any solution predicted below this yield (%),
# no matter how cheap. Set to None to disable.
MIN_YIELD_FLOOR = 85

print("═" * 78)
print("  ECONOMIC MULTI-OBJECTIVE OPTIMISATION  (Derringer–Suich desirability)")
print("═" * 78)

# ── 1. Establish realistic Low/High bounds for Yield & Titer over the design
#      space by dense random sampling of the coded factor space ─────────────
rng_ds = np.random.default_rng(0)
N_SAMPLE = 40000
sample_coded = rng_ds.uniform(-1, 1, size=(N_SAMPLE, K))
sample_df = pd.DataFrame(sample_coded, columns=SN)
sample_df['y'] = 0
sample_yield = model_fit.predict(sample_df).values

acc_idx = FACTOR_NAMES.index(TITER_BASIS)
acc_real_samples = decode(sample_coded[:, acc_idx], TITER_BASIS)
sample_titer = (sample_yield / 100.0) * acc_real_samples

YIELD_LO, YIELD_HI = float(np.clip(sample_yield.min(), 0, None)), float(sample_yield.max())
TITER_LO, TITER_HI = float(np.clip(sample_titer.min(), 0, None)), float(sample_titer.max())

# Cost-driver bounds are simply their design ranges (real units)
IPA_LO, IPA_HI       = FACTORS['Donor']['low'], FACTORS['Donor']['high']
PLP_LO, PLP_HI       = FACTORS['PLP']['low'],   FACTORS['PLP']['high']
ENZYME_LO, ENZYME_HI = FACTORS['Enzyme']['low'], FACTORS['Enzyme']['high']

if IPA_BASIS == 'absolute':
    # Donor_eq × Acceptor spans a wider real range — resample its bounds too
    ipa_abs_samples = decode(sample_coded[:, FACTOR_NAMES.index('Donor')], 'Donor') * acc_real_samples
    IPA_LO, IPA_HI = float(ipa_abs_samples.min()), float(ipa_abs_samples.max())

print(f"  Sampled design-space ranges (n={N_SAMPLE}):")
print(f"    Yield   : {YIELD_LO:6.2f} – {YIELD_HI:6.2f} %")
print(f"    Titer   : {TITER_LO:6.2f} – {TITER_HI:6.2f} mM  (basis: {TITER_BASIS})")
print(f"    IPA     : {IPA_LO:6.3g} – {IPA_HI:6.3g}  ({'eq' if IPA_BASIS=='equivalents' else 'mM, absolute'})")
print(f"    PLP     : {PLP_LO:6.3g} – {PLP_HI:6.3g}  mM")
print(f"    Enzyme  : {ENZYME_LO:6.3g} – {ENZYME_HI:6.3g}  mg/mL\n")

# ── 2. Desirability primitives ────────────────────────────────────────────────
def d_max(y, lo, hi, s=1.0):
    """Desirability for a 'bigger-is-better' response."""
    if hi <= lo: return 1.0
    if y <= lo:  return 0.0
    if y >= hi:  return 1.0
    return ((y - lo) / (hi - lo)) ** s

def d_min(y, lo, hi, s=1.0):
    """Desirability for a 'smaller-is-better' response."""
    if hi <= lo: return 1.0
    if y <= lo:  return 1.0
    if y >= hi:  return 0.0
    return ((hi - y) / (hi - lo)) ** s

def evaluate_point(coded_x):
    """Returns dict of raw predictions + individual desirabilities + overall D."""
    y_pred = predict_from_coded(coded_x)
    reals = {fn: decode(coded_x[i], fn) for i, fn in enumerate(FACTOR_NAMES)}
    acc_real = reals[TITER_BASIS]
    titer = (y_pred / 100.0) * acc_real
    ipa_val = reals['Donor'] * (acc_real if IPA_BASIS == 'absolute' else 1.0)

    dY  = d_max(y_pred, YIELD_LO, YIELD_HI)
    dT  = d_max(titer,  TITER_LO, TITER_HI)
    dI  = d_min(ipa_val, IPA_LO, IPA_HI)
    dP  = d_min(reals['PLP'], PLP_LO, PLP_HI)
    dE  = d_min(reals['Enzyme'], ENZYME_LO, ENZYME_HI)

    if MIN_YIELD_FLOOR is not None and y_pred < MIN_YIELD_FLOOR:
        D = 0.0
    else:
        ws = np.array([IMPORTANCE['Yield'], IMPORTANCE['Titer'], IMPORTANCE['IPA'],
                        IMPORTANCE['PLP'], IMPORTANCE['Enzyme']], dtype=float)
        ds = np.array([dY, dT, dI, dP, dE])
        ds_safe = np.clip(ds, 1e-12, 1.0)  # avoid 0**w log issues
        D = float(np.prod(ds_safe ** (ws / ws.sum())))
        if np.any(ds == 0):
            D = 0.0

    return dict(y_pred=y_pred, titer=titer, ipa_val=ipa_val, reals=reals,
                dY=dY, dT=dT, dI=dI, dP=dP, dE=dE, D=D)

def neg_D(coded_x):
    return -evaluate_point(coded_x)['D']

# ── 3. Global optimisation of overall desirability ────────────────────────────
bounds_econ = [(-1, 1)] * K
de_econ = differential_evolution(neg_D, bounds_econ, seed=42,
                                   maxiter=5000, tol=1e-10, polish=True)
econ_coded = de_econ.x
econ = evaluate_point(econ_coded)

# ── 4. Report: side-by-side vs the yield-only and titer-only optimums ─────────
yonly = evaluate_point(opt_coded)
tonly = evaluate_point(opt_titer_coded)  # Evaluates the Titer optimum

print("─" * 95)
print(f"  {'':<18}{'YIELD OPTIMUM':>22}{'TITER OPTIMUM':>24}{'ECONOMIC OPTIMUM':>26}")
print("─" * 95)
for fn in FACTOR_NAMES:
    unit = FACTORS[fn]['unit']
    print(f"  {fn:<15}{yonly['reals'][fn]:>18.4g} {unit:<4}{tonly['reals'][fn]:>19.4g} {unit:<4}{econ['reals'][fn]:>21.4g} {unit}")
print("-" * 95)
print(f"  {'Predicted Yield (%)':<21}{yonly['y_pred']:>19.2f}{tonly['y_pred']:>24.2f}{econ['y_pred']:>26.2f}")
print(f"  {'Predicted Titer (mM)':<21}{yonly['titer']:>19.2f}{tonly['titer']:>24.2f}{econ['titer']:>26.2f}")
print(f"  {'Overall Desirability':<21}{yonly['D']:>19.3f}{tonly['D']:>24.3f}{econ['D']:>26.3f}")
print("─" * 95)

# Calculate savings/deltas vs Yield-Only Optimum
ipa_save_y    = 100 * (yonly['ipa_val'] - econ['ipa_val']) / yonly['ipa_val'] if yonly['ipa_val'] else 0
plp_save_y    = 100 * (yonly['reals']['PLP'] - econ['reals']['PLP']) / yonly['reals']['PLP'] if yonly['reals']['PLP'] else 0
enz_save_y    = 100 * (yonly['reals']['Enzyme'] - econ['reals']['Enzyme']) / yonly['reals']['Enzyme'] if yonly['reals']['Enzyme'] else 0
yield_delta_y = econ['y_pred'] - yonly['y_pred']
titer_delta_y = econ['titer']  - yonly['titer']

# Calculate savings/deltas vs Titer-Only Optimum
ipa_save_t    = 100 * (tonly['ipa_val'] - econ['ipa_val']) / tonly['ipa_val'] if tonly['ipa_val'] else 0
plp_save_t    = 100 * (tonly['reals']['PLP'] - econ['reals']['PLP']) / tonly['reals']['PLP'] if tonly['reals']['PLP'] else 0
enz_save_t    = 100 * (tonly['reals']['Enzyme'] - econ['reals']['Enzyme']) / tonly['reals']['Enzyme'] if tonly['reals']['Enzyme'] else 0
yield_delta_t = econ['y_pred'] - tonly['y_pred']
titer_delta_t = econ['titer']  - tonly['titer']

print(f"  Moving from Yield Optimum to Economic Optimum:")
print(f"    IPA use      : {ipa_save_y:+.1f} %")
print(f"    PLP use      : {plp_save_y:+.1f} %")
print(f"    Enzyme use   : {enz_save_y:+.1f} %")
print(f"    Yield change : {yield_delta_y:+.2f} percentage points")
print(f"    Titer change : {titer_delta_y:+.2f} mM")
print("-" * 95)
print(f"  Moving from Titer Optimum to Economic Optimum:")
print(f"    IPA use      : {ipa_save_t:+.1f} %")
print(f"    PLP use      : {plp_save_t:+.1f} %")
print(f"    Enzyme use   : {enz_save_t:+.1f} %")
print(f"    Yield change : {yield_delta_t:+.2f} percentage points")
print(f"    Titer change : {titer_delta_t:+.2f} mM")
print("═" * 95)

econ_real = econ['reals']

# ── 5. Diagnostic plots (PUBLICATION READY STYLING) ────────────────────────────
fig_econ = plt.figure(figsize=(18, 8), facecolor='#FFFFFF', edgecolor='#FFFFFF', layout='constrained')
gs = fig_econ.add_gridspec(1, 2, width_ratios=[1, 1.2])

ax_bar = fig_econ.add_subplot(gs[0])
ax_contour = fig_econ.add_subplot(gs[1])
ax_bar.set_facecolor('#FFFFFF')
ax_contour.set_facecolor('#FFFFFF')

# (a) Desirability components bar chart
comp_labels = ['Yield', 'Titer', 'IPA↓', 'PLP↓', 'Enzyme↓']
comp_vals   = [econ['dY'], econ['dT'], econ['dI'], econ['dP'], econ['dE']]
# Safely fallback to hardcoded colors in case TEAL/AMBER/RED_C aren't defined in your global scope
comp_colors = ['#2c7bb6', '#2c7bb6', '#fdae61', '#fdae61', '#d7191c']
bars = ax_bar.bar(comp_labels, comp_vals, color=comp_colors, edgecolor='black', linewidth=1.2)
for b, v in zip(bars, comp_vals):
    ax_bar.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}",
                ha='center', fontsize=16, color='black', fontweight='bold')

ax_bar.axhline(econ['D'], color='black', linestyle='--', linewidth=2,
               label=f"Overall D = {econ['D']:.3f}")
ax_bar.set_ylim(0, 1.15)
ax_bar.set_ylabel("Desirability Score (0–1)", fontsize=18, labelpad=12)
ax_bar.set_title("Economic Optimum: Desirability Breakdown", fontsize=22, pad=15)
ax_bar.tick_params(axis='both', labelsize=16)
ax_bar.legend(fontsize=14, loc='upper right')

# (b) Contour Map
fixed_for_contour = {fn: econ_coded[i] for i, fn in enumerate(FACTOR_NAMES)
                     if fn not in ('Donor', 'Acceptor')}
n_grid_d = 80
xi = np.linspace(-1, 1, n_grid_d)
yi = np.linspace(-1, 1, n_grid_d)
XI, YI = np.meshgrid(xi, yi)
D_grid = np.zeros_like(XI)
for ii in range(n_grid_d):
    for jj in range(n_grid_d):
        cx = np.array([fixed_for_contour.get(fn, 0.0) for fn in FACTOR_NAMES])
        cx[FACTOR_NAMES.index('Donor')] = XI[ii, jj]
        cx[FACTOR_NAMES.index('Acceptor')] = YI[ii, jj]
        D_grid[ii, jj] = evaluate_point(cx)['D']

XI_real = decode(XI, 'Donor')
YI_real = decode(YI, 'Acceptor')

cp = ax_contour.contourf(XI_real, YI_real, D_grid, levels=25, cmap=CMAP_RS, alpha=0.9)
cs = ax_contour.contour(XI_real, YI_real, D_grid, levels=10, colors='white', linewidths=0.8, alpha=0.6)
ax_contour.clabel(cs, fmt='%.2f', fontsize=14, colors='white')

cb = plt.colorbar(cp, ax=ax_contour, pad=0.03)
cb.set_label('Overall desirability D', fontsize=18, labelpad=15)
cb.ax.tick_params(labelsize=14)

# Markers
ax_contour.scatter([econ_real['Donor']], [econ_real['Acceptor']], marker='*', s=800,
                    color='gold', edgecolors='black', linewidths=1.5, zorder=10,
                    label='Economic optimum')
ax_contour.scatter([yonly['reals']['Donor']], [yonly['reals']['Acceptor']], marker='o', s=200,
                    color='white', edgecolors='black', linewidths=2.0, zorder=10,
                    label='Yield-only optimum')

ax_contour.set_xlabel(f"Donor (eq)", fontsize=20, labelpad=12)
ax_contour.set_ylabel(f"Acceptor (mM)", fontsize=20, labelpad=12)
ax_contour.set_title("Desirability Landscape\n(PLP, Enzyme fixed at economic optimum)", fontsize=22, pad=15)
ax_contour.tick_params(axis='both', labelsize=16)
ax_contour.legend(fontsize=14, loc='lower right')

plt.savefig('economic_optimum.png', dpi=300, bbox_inches='tight', facecolor='#FFFFFF', edgecolor='#FFFFFF', transparent=False)
plt.show()
print("✓  Saved economic_optimum.png (Pure White, Publication Style)")

# ── 6. Append to the summary report file ──────────────────────────────────────
econ_lines = [
    "",
    "═" * 85,
    "  MULTI-OBJECTIVE COMPARISON (YIELD vs TITER vs ECONOMIC)",
    "═" * 85,
    f"  Economic Importance weights: {IMPORTANCE}",
    f"  Yield floor constraint: {MIN_YIELD_FLOOR}",
    "",
    f"  {'':<18}{'YIELD OPTIMUM':>20}{'TITER OPTIMUM':>22}{'ECONOMIC OPTIMUM':>22}"
]

for fn in FACTOR_NAMES:
    unit = FACTORS[fn]['unit']
    econ_lines.append(f"  {fn:<15}{yonly['reals'][fn]:>14.4g} {unit:<4}{tonly['reals'][fn]:>17.4g} {unit:<4}{econ['reals'][fn]:>17.4g} {unit}")

econ_lines += [
    f"  {'Yield (%)':<19}{yonly['y_pred']:>15.2f}{tonly['y_pred']:>22.2f}{econ['y_pred']:>22.2f}",
    f"  {'Titer (mM)':<19}{yonly['titer']:>15.2f}{tonly['titer']:>22.2f}{econ['titer']:>22.2f}",
    f"  {'Overall D':<19}{yonly['D']:>15.3f}{tonly['D']:>22.3f}{econ['D']:>22.3f}",
    "",
    f"  Economic Impact (vs. Yield-only optimum):",
    f"  IPA {ipa_save:+.1f}%, PLP {plp_save:+.1f}%, Enzyme {enz_save:+.1f}%",
    f"  Yield {yield_delta:+.2f} pp, Titer {titer_delta:+.2f} mM",
]

with open("RSM_summary_report.txt", "a") as fh:
    fh.write("\n".join(econ_lines))
print("✓  Multi-objective comparison appended to RSM_summary_report.txt")

try:
    display(FileLink('economic_optimum.png', result_html_prefix="⬇  economic_optimum.png: "))
except Exception:
    pass

## Interactive 3D response surfaces: automatically scaled axis

The next cell uses Plotly to generate rotatable, zoomable response surfaces with experimental observations overlaid. The response-axis range is determined automatically for each plot.

Run it after the core model because it uses `FACTOR_PAIRS`, `predict_grid`, `MODEL_DF`, and `RESPONSE_NAME`. No experimental inputs are entered in this cell. Change `custom_ryg` only if a different color scale is desired.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §8B — INTERACTIVE 3D RESPONSE SURFACES (PLOTLY)
#   Generates fully interactive 3D plots using your original Red-Yellow-Green cmap.
# ══════════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go

print("Generating Interactive 3D Surface Plots...")
print("Click and drag to rotate, scroll to zoom. Hover over points for exact values.\n")

# Define your custom Red-Yellow-Green colorscale for Plotly
custom_ryg = [
    [0.00, "#C0392B"], # Red
    [0.25, "#E67E22"], # Orange
    [0.50, "#F1C40F"], # Yellow
    [0.75, "#2ECC71"], # Light Green
    [1.00, "#1A8A4A"]  # Dark Green
]

# Loop through all pairs of factors
for fn_x, fn_y in FACTOR_PAIRS:
    # Get the surface grid from your existing predict_grid function
    XI, YI, ZI = predict_grid(fn_x, fn_y)

    # Get the actual measured data points to overlay on the plot
    x_act = [decode(MODEL_DF[short[fn_x]].iloc[i], fn_x) for i in range(len(MODEL_DF))]
    y_act = [decode(MODEL_DF[short[fn_y]].iloc[i], fn_y) for i in range(len(MODEL_DF))]
    z_act = MODEL_DF['y'].values

    fig = go.Figure()

    # 1. Add the 3D surface with the custom colormap
    fig.add_trace(go.Surface(
        x=XI, y=YI, z=ZI,
        colorscale=custom_ryg,
        opacity=0.88,
        name='Predicted Surface',
        colorbar=dict(title=RESPONSE_NAME, len=0.6, thickness=15, x=0.9)
    ))

    # 2. Add the actual measured data points as black spheres
    fig.add_trace(go.Scatter3d(
        x=x_act, y=y_act, z=z_act,
        mode='markers',
        marker=dict(
            size=5,
            color='black',
            line=dict(color='white', width=1)
        ),
        name='Measured Data'
    ))

    # 3. Format the layout and axes
    fx = FACTORS[fn_x]
    fy = FACTORS[fn_y]

    fig.update_layout(
        title=dict(
            text=f"Interactive Surface: <b>{fn_x} × {fn_y}</b>",
            x=0.5, y=0.95
        ),
        scene=dict(
            xaxis_title=f"{fn_x} ({fx['unit']})",
            yaxis_title=f"{fn_y} ({fy['unit']})",
            zaxis_title=RESPONSE_NAME,
            xaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            yaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            zaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
        ),
        width=800,
        height=650,
        margin=dict(l=20, r=20, b=20, t=60),
        paper_bgcolor="#F8F9FA",
    )

    # Show the plot right in the output
    fig.show()

## Interactive 3D response surfaces: fixed 0–100% axis

This final cell is an alternative interactive visualization that fixes the response axis at 0–100%, making surfaces easier to compare across factor pairs when the response is percentage yield.

Use this version when `RESPONSE_NAME` is a percentage bounded by 0 and 100. If the response has different units or limits, the fixed axis is not appropriate. Running both interactive cells is safe but produces two sets of similar plots; choose the version that best suits the intended presentation.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# §8B — INTERACTIVE 3D RESPONSE SURFACES (PLOTLY)
#   Generates fully interactive 3D plots using your original Red-Yellow-Green cmap
#   and a fixed 0-100% Z-axis for perfect cross-plot comparison.
# ══════════════════════════════════════════════════════════════════════════════

import plotly.graph_objects as go

print("Generating Interactive 3D Surface Plots...")
print("Click and drag to rotate, scroll to zoom. Hover over points for exact values.\n")

# Define your custom Red-Yellow-Green colorscale for Plotly
custom_ryg = [
    [0.00, "#C0392B"], # Red
    [0.25, "#E67E22"], # Orange
    [0.50, "#F1C40F"], # Yellow
    [0.75, "#2ECC71"], # Light Green
    [1.00, "#1A8A4A"]  # Dark Green
]

# Loop through all pairs of factors
for fn_x, fn_y in FACTOR_PAIRS:
    # Get the surface grid from your existing predict_grid function
    XI, YI, ZI = predict_grid(fn_x, fn_y)

    # Get the actual measured data points to overlay on the plot
    x_act = [decode(MODEL_DF[short[fn_x]].iloc[i], fn_x) for i in range(len(MODEL_DF))]
    y_act = [decode(MODEL_DF[short[fn_y]].iloc[i], fn_y) for i in range(len(MODEL_DF))]
    z_act = MODEL_DF['y'].values

    fig = go.Figure()

    # 1. Add the 3D surface with the custom colormap and LOCKED color range
    fig.add_trace(go.Surface(
        x=XI, y=YI, z=ZI,
        colorscale=custom_ryg,
        cmin=0,      # Locks the bottom of the color scale to 0%
        cmax=100,    # Locks the top of the color scale to 100%
        opacity=0.88,
        name='Predicted Surface',
        colorbar=dict(title=RESPONSE_NAME, len=0.6, thickness=15, x=0.9)
    ))

    # 2. Add the actual measured data points as black spheres
    fig.add_trace(go.Scatter3d(
        x=x_act, y=y_act, z=z_act,
        mode='markers',
        marker=dict(
            size=5,
            color='black',
            line=dict(color='white', width=1)
        ),
        name='Measured Data'
    ))

    # 3. Format the layout and axes
    fx = FACTORS[fn_x]
    fy = FACTORS[fn_y]

    fig.update_layout(
        title=dict(
            text=f"Interactive Surface: <b>{fn_x} × {fn_y}</b>",
            x=0.5, y=0.95
        ),
        scene=dict(
            xaxis_title=f"{fn_x} ({fx['unit']})",
            yaxis_title=f"{fn_y} ({fy['unit']})",
            zaxis_title=RESPONSE_NAME,
            xaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            yaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            zaxis=dict(
                backgroundcolor="white",
                gridcolor="lightgrey",
                range=[0, 100]  # Locks the physical Z-axis to 0-100
            ),
        ),
        width=800,
        height=650,
        margin=dict(l=20, r=20, b=20, t=60),
        paper_bgcolor="#F8F9FA",
    )

    # Show the plot right in the output
    fig.show()